### Conception d'une base de donnée NoSql distribuée
1. Chargement de la base  
2. Analyse de l'imbrication

In [2]:
import os
import polars as pl 
from dotenv import load_dotenv
from pymongo import MongoClient 

In [3]:
# On recupere les credentials qu'on initialise dans une variable (uri)

load_dotenv()
uri = os.getenv("NONGO_URI")

In [4]:
# On se connecte au serveur mongodb (Le container doit tourner)

client = MongoClient(uri)
db = client['airbnb']
collection = db['annonces']

In [5]:
# Test du retour serveur
client.admin. command('ping')

{'ok': 1.0}

In [ ]:
# On extrait un csv de la liste des colonnes pour analyser imbrications

docs = list(collection.find({}, {'_id':0})) # Extrait la liste des colonnes 
df = pl.DataFrame(docs,  infer_schema_length=None) # Créer un df à partir de la liste, et scanne toute les lignes pour detecter les types
df = df.with_columns(pl.col(pl.String).replace("", None)) # Remplace les chaines vide en null

def renseigne(df): # Fonctioin de profiling adapter à polars
    n = df.height
    nulls = df.null_count().row(0)
    profil = pl.DataFrame({
        'colonne': df.columns, 
        'type':[str(t)for t in df.dtypes],
        'longueur_max':[df[c].cast(pl.Utf8).str.len_chars().max() for c in df.columns],
        'non_null':[n - x for x in nulls],
        'nan':list(nulls),
        "% nan": [round(x / n * 100, 2) for x in nulls],
        'valeur_unique':[df[c].n_unique()for c in df.columns]
    })
    return profil

cols_airbnb = renseigne(df) # Instance le df profilé
cols_airbnb.write_csv('../docs/cols_airbnb.csv') # Produit un csv du profil
display(cols_airbnb) # Affiche le résultat de la fonctioin de profiling

colonne,type,longueur_max,non_null,nan,% nan,valeur_unique
str,str,i64,i64,i64,f64,i64
"""id""","""Int64""",19,95885,0,0.0,95885
"""listing_url""","""String""",48,95885,0,0.0,95885
"""scrape_id""","""Int64""",14,95885,0,0.0,1
"""last_scraped""","""String""",10,95885,0,0.0,4
"""source""","""String""",15,95885,0,0.0,2
…,…,…,…,…,…,…
"""calculated_host_listings_count""","""Int64""",3,95885,0,0.0,99
"""calculated_host_listings_count…","""Int64""",3,95885,0,0.0,99
"""calculated_host_listings_count…","""Int64""",3,95885,0,0.0,24
